# Spec-Craft with OpenCode: Interactive Agent & MCP Development

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_opencode.ipynb)

Welcome to the **Spec-Craft & OpenCode** interactive development tutorial on Google Colab!

This tutorial guides you through using **OpenCode** (an autonomous terminal AI coding agent) with **Spec-Craft**'s native FastMCP server and **spec-kit** workflows directly inside Google Colab via an interactive terminal.

### Architecture Overview
```mermaid
flowchart TB
    subgraph Host["Google Colab / Linux Host"]
        subgraph UI["Developer Interface"]
            ColabXterm["Interactive Terminal (colab-xterm)"]
        end

        subgraph AgentEnv["OpenCode Agent"]
            OpenCodeCLI["opencode CLI"]
        end

        subgraph LocalLLM["Local Inference Engine"]
            OllamaServer["Ollama Daemon (num_ctx=32768)"]
            GemmaModel["Gemma 4 QAT"]
            OllamaServer --> GemmaModel
        end

        subgraph SpecCraftMCP["Spec-Craft FastMCP Server"]
            MCPServer["spec-craft serve"]
            StoryboardParser["StoryboardParser (Obsidian)"]
            Detector["WorkspaceDetector (spec-kit)"]
            MCPServer --> StoryboardParser
            MCPServer --> Detector
        end
        
        ColabXterm -->|User Prompts| OpenCodeCLI
        OpenCodeCLI -->|OpenAI-Compatible API /v1| OllamaServer
        OpenCodeCLI -->|MCP Protocol (JSON-RPC stdio)| MCPServer
    end
```

### Protocol Sequence
```mermaid
sequenceDiagram
    autonumber
    actor Dev as Developer
    participant Terminal as colab-xterm
    participant OpenCode as OpenCode Agent
    participant MCP as spec-craft FastMCP Server
    participant Ollama as Ollama (Gemma 4 QAT)

    Dev->>Terminal: specify init --here --force --integration opencode
    Dev->>Terminal: script -q -c 'opencode' /content/opencode-session.log
    OpenCode->>MCP: Handshake (JSON-RPC stdio)
    MCP-->>OpenCode: Tools: spec-craft_check_environment, spec-craft_read_storyboard
    Dev->>OpenCode: Task 1: Check Environment Tool
    OpenCode->>MCP: spec-craft_check_environment()
    MCP-->>OpenCode: Environment Status JSON
    Dev->>OpenCode: Task 2: Draft Strategy (obsidian/strategy_task_tracker.md)
    OpenCode->>Ollama: Generate Markdown Specification
    OpenCode->>Dev: File Created
    Dev->>OpenCode: Task 3: Inspect Strategy with MCP
    OpenCode->>MCP: spec-craft_read_storyboard(file_path)
    MCP-->>OpenCode: Sections & Metadata
    Dev->>OpenCode: Task 4: Implement task_tracker.py & test_task_tracker.py
    OpenCode->>Ollama: Generate Implementation & Pytest Suite
    Dev->>Terminal: pytest test_task_tracker.py -v
```


In [ ]:
# ==============================================================================
# 1. Environment Setup (Node.js, OpenCode, colab-xterm, Ollama, spec-craft)
# ==============================================================================
import os, sys, time, subprocess, json

print("📦 1. Installing system dependencies & Node.js 20 LTS...")
!apt-get update -qq && apt-get install -y -qq curl git build-essential zstd pkg-config libssl-dev > /dev/null
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null

print("📥 2. Installing OpenCode CLI, colab-xterm, and uv...")
!npm install -g opencode-ai
!pip install -q colab-xterm uv pytest

print("🚀 3. Installing and launching Ollama background service (num_ctx=32768)...")
!curl -fsSL https://ollama.com/install.sh | sh

# Configure Ollama context length and start service in background
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
ollama_env = os.environ.copy()
subprocess.Popen(["ollama", "serve"], env=ollama_env)

# Health check: wait for Ollama daemon to respond
for _ in range(12):
    time.sleep(1)
    try:
        res = subprocess.run(["curl", "-s", "http://127.0.0.1:11434"], capture_output=True, text=True)
        if "Ollama is running" in res.stdout:
            print("   ✅ Ollama server is active (context length: 32768)!")
            break
    except Exception:
        pass

print("🧠 4. Pulling Ollama model (Gemma 4 QAT)...")
# Model Choices:
# [Recommended Default for T4 GPU] 12B QAT (~7.2 GB VRAM):
!ollama pull gemma4:12b-it-qat
# --- Lightweight alternatives for Colab Free Tier / CPU ---
# [Fast / Low VRAM] 4B QAT (~3.8 GB VRAM):
# !ollama pull gemma4:e4b-it-qat
# [Ultra-Lightweight] 2B QAT (~2.2 GB VRAM):
# !ollama pull gemma4:e2b-it-qat

print("🛠️ 5. Installing specify-cli (spec-kit) & spec-craft...")
!uv tool install specify-cli || uv tool install specify-cli --force --from git+https://github.com/github/spec-kit.git
!uv pip install --system git+https://github.com/akuroiwa/spec-craft.git

# Set global PATH for current Python environment
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

print("🎉 Core environment ready!")


## 2. Configure OpenCode MCP Settings (`opencode.json`)

OpenCode supports local MCP servers via the `mcp` field in `~/.config/opencode/opencode.json` (see [OpenCode Configuration Docs](https://opencode.ai/v2/docs/config/) and [MCP Servers Docs](https://opencode.ai/v2/docs/mcp-servers/)).

In OpenCode v2:
- The local MCP server `command` is an array of strings: `["uv", "run", "spec-craft", "serve"]`.
- The local Ollama provider uses `@ai-sdk/openai-compatible` pointing to `http://127.0.0.1:11434/v1`.


In [ ]:
# ==============================================================================
# Dynamic OpenCode Configuration (Ollama Provider + spec-craft MCP Server)
# ==============================================================================
config_dir = os.path.expanduser("~/.config/opencode")
os.makedirs(config_dir, exist_ok=True)

opencode_config = {
  "$schema": "https://opencode.ai/config.json",
  "provider": {
    "ollama": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "Ollama (local)",
      "options": {
        "baseURL": "http://127.0.0.1:11434/v1"
      },
      "models": {
        "gemma4:12b-it-qat": {
          "name": "Gemma 4 12B QAT"
        }
      }
    }
  },
  "model": "ollama/gemma4:12b-it-qat",
  "mcp": {
    "spec-craft": {
      "type": "local",
      "command": [
        "uv",
        "run",
        "spec-craft",
        "serve"
      ]
    }
  }
}

config_path = os.path.join(config_dir, "opencode.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(opencode_config, f, indent=2)

# Also place a copy in project root for local execution
with open("opencode.json", "w", encoding="utf-8") as f:
    json.dump(opencode_config, f, indent=2)

print(f"✅ OpenCode MCP configuration written to {config_path} and ./opencode.json")


## 3. Verify Prerequisites

Check that all environment tools and MCP bridges are ready using `spec-craft check-env`.


In [ ]:
!spec-craft check-env


## 4. Step-by-Step OpenCode Task Protocol

> **Important (Strict Scoping & Avoiding Looping)**:
> Local models like Gemma 4 require explicit boundary instructions. Always include `Do not ask questions.` and `Do not inspect unrelated files.` in the prompts below to prevent conversational questioning loops:

| Step | Phase | OpenCode Prompt to Enter (Copy & Paste) |
| :--- | :--- | :--- |
| **Task 1** | **MCP Tool Verification** | `"Use the spec-craft_check_environment tool to verify environment readiness. Do not ask questions. Do not inspect unrelated files. Do not modify files. Summarize the tool result."` |
| **Task 2** | **Strategy Drafting** | `"Create exactly one file 'obsidian/strategy_task_tracker.md' with YAML frontmatter (title: 'Markdown Task Tracker CLI', tags: [sdd, cli, utility], status: draft) and H1 sections (# 1. Overview, # 2. User Stories, # 3. Technical Requirements). Do not ask questions. Do not modify other files."` |
| **Task 3** | **Strategy Inspection (MCP)**| `"Use the spec-craft_read_storyboard tool to inspect 'obsidian/strategy_task_tracker.md'. Do not ask questions. Do not modify any file. Report sections and metadata found."` |
| **Task 4** | **Implementation** | `"Implement exactly one file 'task_tracker.py' with TaskTracker class and analyze_tasks(content: str) -> dict function returning {'total': int, 'completed': int, 'pending': int, 'percentage': float}. Do not ask questions. Do not modify other files."` |
| **Task 5** | **Testing & Verification** | `"Create exactly one file 'test_task_tracker.py' with exactly 4 pytest unit tests (empty string, all pending, all completed, mixed tasks). Do not ask questions. Run pytest test_task_tracker.py."` |


## 5. Interactive Web Terminal with Session Recording (`colab-xterm`)

Execute the cell below to open an interactive Linux terminal within Google Colab.

### Terminal Execution Steps:
```bash
# 1. Initialize spec-kit in current directory
specify init --here --force --integration opencode

# 2. Ensure Ollama context length is set
export OLLAMA_CONTEXT_LENGTH=32768

# 3. Launch OpenCode inside 'script' to record full terminal session logs
script -q -c 'opencode' /content/opencode-session.log
```

Inside OpenCode TUI, enter the step-by-step prompts from Section 4 above.


In [ ]:
# Load colabxterm extension and launch interactive terminal
%load_ext colabxterm
%xterm height=550


## 6. Verification & Test Execution

After completing tasks in OpenCode, verify that all unit tests pass directly in the notebook.


In [ ]:
# Execute unit tests
!pytest test_task_tracker.py -v


## 7. Inspect & Download OpenCode Session Logs

Review the recorded terminal log file (`opencode-session.log`) and download it to your local machine for analysis and reporting.


In [ ]:
# List and inspect generated log files
from pathlib import Path

log_files = list(Path("/content").glob("*.log"))
print(f"📁 Found {len(log_files)} log file(s):")
for log_path in log_files:
    print(f"   - {log_path.name}: {log_path.stat().st_size:,} bytes")

# Uncomment to download log file to your local computer:
# from google.colab import files
# if Path("/content/opencode-session.log").exists():
#     files.download("/content/opencode-session.log")


## 8. YouTube Video Demo & Recording Guide

For creating YouTube demonstration videos of Spec-Craft & OpenCode, here are recommended approaches:

### A. Screen Recording Tools
- **Screenity / Loom (Chrome Extension)**: Capture clean, high-resolution browser tabs with mouse click ripple effects and audio commentary.
- **OBS Studio**: Capture window or full screen with custom overlays, zoom effects, and high bitrate recording.

### B. Programmatic Automated Recording (Playwright)
You can also automate browser execution and record video using Playwright:
```python
# Example: Automated recording with Playwright
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    context = browser.new_context(record_video_dir="videos/", record_video_size={"width": 1280, "height": 720})
    page = context.new_page()
    page.goto("https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_opencode.ipynb")
    # Automate actions...
    context.close()
```

## Summary & Next Steps

Congratulations! You have experienced full interactive Spec-Driven Development with Spec-Craft & OpenCode:
- **Strategy (Obsidian)**: Captured user stories and technical requirements.
- **MCP Server (spec-craft)**: Exposed tools (`StoryboardParser`, `check-env`) directly to OpenCode.
- **Interactive TUI (OpenCode)**: Collaborated with the AI agent in real time with step-by-step execution.
- **Execution Logs**: Recorded complete session logs via `script` for review and verification.
- **Tactics (pytest)**: Verified correctness through automated test suites.

### Resources
- **GitHub Repository**: [https://github.com/akuroiwa/spec-craft](https://github.com/akuroiwa/spec-craft)
- **OpenCode Documentation**: [https://opencode.ai/v2/docs/mcp-servers/](https://opencode.ai/v2/docs/mcp-servers/)
- **Documentation**: [https://spec-craft.readthedocs.io/](https://spec-craft.readthedocs.io/)
- **Zenn Book / Articles**: [spec-craft Development Notes](https://zenn.dev/akuroiwa/books/dev-note-spec-craft)
